<i>Copyright (c) Microsoft Corporation. All rights reserved.</i>

<i>Licensed under the MIT License.</i>

# Neural Collaborative Filtering on MovieLens dataset.

Neural Collaborative Filtering (NCF) is a well known recommendation algorithm that generalizes the matrix factorization problem with multi-layer perceptron. 

This notebook provides an example of how to utilize and evaluate NCF implementation in the `recommenders`. We use a smaller dataset in this example to run NCF efficiently with GPU acceleration on a [Data Science Virtual Machine](https://azure.microsoft.com/en-gb/services/virtual-machines/data-science-virtual-machines/).

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import pandas as pd
import tensorflow as tf
tf.get_logger().setLevel('ERROR') # only show error messages

from recommenders.utils.timer import Timer
from recommenders.models.ncf.ncf_singlenode import NCF
from recommenders.models.ncf.dataset import Dataset as NCFDataset
from recommenders.datasets import movielens
from recommenders.utils.notebook_utils import is_jupyter
from recommenders.datasets.python_splitters import python_chrono_split, python_random_split
from recommenders.evaluation.python_evaluation import (rmse, mae, rsquared, exp_var, map_at_k, ndcg_at_k, precision_at_k, 
                                                     recall_at_k, get_top_k_items)

print("System version: {}".format(sys.version))
print("Pandas version: {}".format(pd.__version__))
print("Tensorflow version: {}".format(tf.__version__))

System version: 3.8.16 | packaged by conda-forge | (default, Feb  1 2023, 16:01:55) 
[GCC 11.3.0]
Pandas version: 1.5.3
Tensorflow version: 2.7.4


Set the default parameters.

In [3]:
# top k items to recommend
TOP_K = 10

# Select MovieLens data size: 100k, 1m, 10m, or 20m
# Custom selects optimizer dataset
MOVIELENS_DATA_SIZE = '100k'

# Model parameters
EPOCHS = 50
BATCH_SIZE = 256

SEED = 42

### 1. Download the MovieLens dataset

In [4]:
if MOVIELENS_DATA_SIZE == 'custom':
    df = pd.read_csv('optimizer_dataset.csv', index_col=0)
else:
    df = movielens.load_pandas_df(
        size=MOVIELENS_DATA_SIZE,
        header=["userID", "itemID", "rating", "timestamp"]
    )

INFO:recommenders.datasets.download_utils:Downloading https://files.grouplens.org/datasets/movielens/ml-100k.zip
100%|██████████| 4.81k/4.81k [00:04<00:00, 1.17kKB/s]


### 2. Split the data using the Spark chronological splitter provided in utilities

In [5]:
train, test = python_random_split(df, 0.75) # python_chrono_split

# Sort by user
train.sort_values(by=["userID"], inplace=True)
test.sort_values(by=["userID"], inplace=True)

Filter out any users or items in the test set that do not appear in the training set.

In [6]:
test = test[test["userID"].isin(train["userID"].unique())]
test = test[test["itemID"].isin(train["itemID"].unique())]
test

,userID,itemID,rating,timestamp
25255,1,32,5.0,888732909
22496,1,68,4.0,875072688
78171,1,58,4.0,878542960
8844,1,262,3.0,875071421
48078,1,238,4.0,875072235
...,...,...,...,...
96983,943,401,1.0,888639867
95132,943,67,4.0,888640143
91550,943,231,2.0,888640186
94028,943,732,4.0,888639789


Write datasets to csv files.

In [7]:
train_file = "./train.csv"
test_file = "./test.csv"
train.to_csv(train_file, index=False)
test.to_csv(test_file, index=False)

Generate an NCF dataset object from the data subsets.

In [8]:
data = NCFDataset(train_file=train_file, test_file=test_file, seed=SEED)

INFO:recommenders.models.ncf.dataset:Indexing ./train.csv ...
INFO:recommenders.models.ncf.dataset:Indexing ./test.csv ...
INFO:recommenders.models.ncf.dataset:Indexing ./test_full.csv ...


### 3. Train the NCF model on the training data, and get the top-k recommendations for our testing data

NCF accepts implicit feedback and generates prospensity of items to be recommended to users in the scale of 0 to 1. A recommended item list can then be generated based on the scores. Note that this quickstart notebook is using a smaller number of epochs to reduce time for training. As a consequence, the model performance will be slighlty deteriorated. 

In [9]:
model = NCF (
    n_users=data.n_users, 
    n_items=data.n_items,
    model_type="NeuMF",
    n_factors=4,
    layer_sizes=[16,8,4],
    n_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=1e-3,
    verbose=10,
    seed=SEED
)

/home/martin/miniconda3/envs/l2co/lib/python3.8/site-packages/tensorflow/python/keras/engine/base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '
2023-02-23 11:52:48.846299: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-02-23 11:52:56.500084: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2023-02-23 11:52:56.500167: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (martin): /proc/driver/nvidia/version does not exis

In [10]:
with Timer() as train_time:
    model.fit(data)

print("Took {} seconds for training.".format(train_time))

INFO:recommenders.models.ncf.ncf_singlenode:Epoch 10 [3.63s]: train_loss = 0.288592 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 20 [4.27s]: train_loss = 0.275989 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 30 [3.81s]: train_loss = 0.268655 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 40 [4.50s]: train_loss = 0.262977 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 50 [4.49s]: train_loss = 0.258874 


Took 204.7216 seconds for training.


In the movie recommendation use case scenario, seen movies are not recommended to the users.

In [11]:
with Timer() as test_time:
    users, items, preds = [], [], []
    item = list(train.itemID.unique())
    for user in train.userID.unique():
        user = [user] * len(item) 
        users.extend(user)
        items.extend(item)
        preds.extend(list(model.predict(user, item, is_list=True)))

    all_predictions = pd.DataFrame(data={"userID": users, "itemID":items, "prediction":preds})

    merged = pd.merge(train, all_predictions, on=["userID", "itemID"], how="outer")
    all_predictions = merged[merged.rating.isnull()].drop('rating', axis=1)

print("Took {} seconds for prediction.".format(test_time))

Took 9.2159 seconds for prediction.


### 4. Evaluate how well NCF performs

The ranking metrics are used for evaluation.

In [12]:
eval_map = map_at_k(test, all_predictions, col_prediction='prediction', k=TOP_K)
eval_ndcg = ndcg_at_k(test, all_predictions, col_prediction='prediction', k=TOP_K)
eval_precision = precision_at_k(test, all_predictions, col_prediction='prediction', k=TOP_K)
eval_recall = recall_at_k(test, all_predictions, col_prediction='prediction', k=TOP_K)

print("MAP:\t%f" % eval_map,
      "NDCG:\t%f" % eval_ndcg,
      "Precision@K:\t%f" % eval_precision,
      "Recall@K:\t%f" % eval_recall, sep='\n')

MAP:	0.101176
NDCG:	0.379454
Precision@K:	0.337049
Recall@K:	0.177184


In [13]:
itemid = int(df.iloc[0]['itemID'])

In [14]:
model.predict(user_input=2, item_input=itemid)

0.8034039735794067

In [24]:
df[(df['itemID'] == 276) & (df['userID'] == 2)]

,userID,itemID,rating,timestamp
25334,2,276,4.0,888551552


In [23]:
merged[merged['userID'] == 2]

,userID,itemID,rating,timestamp,prediction
198,2,276,4.0,888551552.0,0.956362
199,2,13,4.0,888551922.0,0.871287
200,2,1,4.0,888550871.0,0.888626
201,2,297,4.0,888550871.0,0.804923
202,2,304,4.0,888979197.0,0.938378
...,...,...,...,...,...
78038,2,1679,NaN,NaN,0.000511
78039,2,1678,NaN,NaN,0.000447
78040,2,1345,NaN,NaN,0.001248
78041,2,1080,NaN,NaN,0.095513


In [20]:
all_predictions

,userID,itemID,timestamp,prediction
75000,1,276,NaN,0.617943
75001,1,1,NaN,0.643997
75002,1,297,NaN,0.059441
75003,1,304,NaN,0.121314
75004,1,289,NaN,0.247867
...,...,...,...,...
1548401,943,1679,NaN,0.000003
1548402,943,1678,NaN,0.000004
1548403,943,1345,NaN,0.000020
1548404,943,1080,NaN,0.000009
